<a href="https://colab.research.google.com/github/iOSRajaramMohanty/GenAI/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📚 RAG Pipeline using LangChain + Chroma + Gemini

This notebook demonstrates how to build a Retrieval-Augmented Generation (RAG) system using:

- LangChain  
- ChromaDB  
- Google Gemini  

## Workflow:
1. Load documents (local)
2. Split into chunks  
3. Generate embeddings  
4. Store in vector DB  
5. Retrieve relevant chunks  
6. Build context  
7. Create prompt  
8. Generate answer using LLM  

## Flow:
Query → Retrieve → Context → Prompt → LLM → Answer  

## Key Idea:
- Combine retrieval with LLM  
- Use context to guide responses  
- Reduce hallucination and improve accuracy  

In [21]:
import sys
import subprocess

# Install necessary libraries
!{sys.executable} -m pip install pypdf python-docx langchain
!pip install -q \
    chromadb \
    langchain \
    langchain-community \
    langchain-google-genai \
    pypdf \
    python-docx

Once the libraries are installed, you can upload your document. The code below will prompt you to select a file from your local machine. It supports PDF, DOCX, and TXT files.

In [22]:
from google.colab import files
import io
import os
from pypdf import PdfReader
from docx import Document

def upload_and_extract_text():
    uploaded = files.upload()
    if not uploaded:
        print("No file uploaded.")
        return None

    file_name = list(uploaded.keys())[0]
    file_content = uploaded[file_name]

    text_content = ""
    file_extension = os.path.splitext(file_name)[1].lower()

    if file_extension == '.pdf':
        try:
            pdf_file = io.BytesIO(file_content)
            reader = PdfReader(pdf_file)
            for page in reader.pages:
                text_content += page.extract_text() or ""
            print(f"Successfully extracted text from PDF: {file_name}")
        except Exception as e:
            print(f"Error processing PDF file {file_name}: {e}")
            return None
    elif file_extension == '.docx':
        try:
            doc_file = io.BytesIO(file_content)
            document = Document(doc_file)
            for para in document.paragraphs:
                text_content += para.text + "\n"
            print(f"Successfully extracted text from DOCX: {file_name}")
        except Exception as e:
            print(f"Error processing DOCX file {file_name}: {e}")
            return None
    elif file_extension == '.txt':
        try:
            text_content = file_content.decode('utf-8')
            print(f"Successfully extracted text from TXT: {file_name}")
        except Exception as e:
            print(f"Error processing TXT file {file_name}: {e}")
            return None
    else:
        print(f"Unsupported file type: {file_extension}. Please upload a PDF, DOCX, or TXT file.")
        return None

    return text_content

raw_document_text = upload_and_extract_text()

if raw_document_text:
    print(f"\nExtracted text snippet (first 500 chars):\n{raw_document_text[:500]}...")
else:
    print("Text extraction failed or no file was uploaded.")

Saving LICENSE_COMPLIANCE.pdf to LICENSE_COMPLIANCE.pdf
Successfully extracted text from PDF: LICENSE_COMPLIANCE.pdf

Extracted text snippet (first 500 chars):
Activepieces License Compliance
Documentation
Reference: Oﬃcial License Documentation
Repository: activepieces/activepieces
This document maps the Activepieces dual-licensing model to this codebase and provides guidance
for compliance when using, modifying, or distributing the software.
1. License Summary
Component License Usage Terms
Core / Community
Edition MIT (Expat) Free to use, modify, and distribute.
Include license notice when distributing.
Enterprise / CloudCommercial
(packages/ee/LICEN...


Now that we have the raw text, we can split it into smaller, more manageable chunks. We'll use `RecursiveCharacterTextSplitter` from `langchain` for this, which tries to split by paragraphs, then sentences, then words, to keep chunks as semantically coherent as possible.

In [23]:
import sys
!{sys.executable} -m pip install langchain_text_splitters

Now that `langchain_text_splitters` is installed, let's re-run the code to split the document into chunks.

In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

if raw_document_text:
    # Initialize the text splitter
    # You can adjust chunk_size and chunk_overlap as needed
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,  # Maximum characters per chunk
        chunk_overlap=200,  # Overlap between chunks to maintain context
        length_function=len,
        is_separator_regex=False,
    )

    # Split the document text into chunks
    document_chunks = text_splitter.create_documents([raw_document_text])

    print(f"\nOriginal text length: {len(raw_document_text)} characters")
    print(f"Number of chunks created: {len(document_chunks)}")
    print("\nFirst chunk example:")
    print(document_chunks[0].page_content)

    # You can now iterate through `document_chunks` to process each chunk
    # for example, embedding them for a vector database.
else:
    print("No text to chunk. Please ensure a document was successfully uploaded and text extracted.")


Original text length: 5548 characters
Number of chunks created: 7

First chunk example:
Activepieces License Compliance
Documentation
Reference: Oﬃcial License Documentation
Repository: activepieces/activepieces
This document maps the Activepieces dual-licensing model to this codebase and provides guidance
for compliance when using, modifying, or distributing the software.
1. License Summary
Component License Usage Terms
Core / Community
Edition MIT (Expat) Free to use, modify, and distribute.
Include license notice when distributing.
Enterprise / CloudCommercial
(packages/ee/LICENSE)
Requires valid Activepieces license for
self-hosted production use.
Third-party
components Original licenses Each component retains its original
license.
2. License Boundaries in the Codebase
2.1 MIT License (Community Edition)
Content outside the following directories is under the MIT license:
Root  LICENSE  ﬁle deﬁnes MIT for all content except:
 packages/ee/  (if exists)
 packages/server/api/src/app/e

In [25]:
import sys
!{sys.executable} -m pip install langchain-google-genai

To use Google's Generative AI embeddings, you'll need an API key. If you don't already have one, create a key in Google AI Studio.
In Colab, add the key to the secrets manager under the "🔑" in the left panel. Give it the name `GOOGLE_API_KEY`. Then, we'll set up the environment.

In [26]:
# Import the Python SDK
import google.generativeai as genai
# Used to securely store your API key
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

Now, let's initialize the embedding model and generate embeddings for each of our document chunks.

In [27]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os

if document_chunks:
    # Set the GOOGLE_API_KEY environment variable
    os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY

    # Initialize the embedding model with the correct model name
    embeddings_model = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

    # Generate embeddings for each chunk
    # The .embed_documents method is designed for lists of strings/documents
    chunk_texts = [chunk.page_content for chunk in document_chunks]
    document_embeddings = embeddings_model.embed_documents(chunk_texts)

    print(f"Generated embeddings for {len(document_embeddings)} chunks.")
    print(f"Example embedding for the first chunk (first 10 dimensions):\n{document_embeddings[0][:10]}...")
    print(f"Dimension of each embedding: {len(document_embeddings[0])}")
else:
    print("No document chunks available to embed. Please ensure text extraction and chunking were successful.")

Generated embeddings for 7 chunks.
Example embedding for the first chunk (first 10 dimensions):
[0.005595682, 0.033704467, 0.007404338, -0.0745742, -0.018533492, 0.021760363, 0.006194696, -0.02083406, 0.014050875, -0.009002757]...
Dimension of each embedding: 3072


Now that `chromadb` and `langchain_community` are installed, let's create a Chroma vector store from our document chunks and their embeddings. We'll also perform a sample similarity search.

In [28]:
import sys
import pkg_resources


# Verify versions
print("\n--- Installed Versions ---")

def print_package_version(package_name):
    try:
        version = pkg_resources.get_distribution(package_name).version
        print(f"{package_name}: {version}")
    except pkg_resources.DistributionNotFound:
        print(f"{package_name}: Not Found")

print_package_version("chromadb")
print_package_version("langchain")
print_package_version("langchain_community")
print("--------------------------")


--- Installed Versions ---
chromadb: 1.5.7
langchain: 1.2.15
langchain_community: 0.4.1
--------------------------


Now that the libraries have been reinstalled, let's try creating the Chroma vector store again.

In [29]:
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

if document_chunks and document_embeddings:
    # Initialize chromadb client explicitly.
    # For in-memory, use chromadb.Client(). For persistent storage, use chromadb.PersistentClient(path="./chroma_db").
    # We'll use an in-memory client for this example.

    vectorstore = Chroma.from_documents(
        documents=document_chunks,
        embedding=embeddings_model,
    )

    print("Chroma vector store created successfully.")

    query = "What are the licensing terms for Activepieces?"
    docs = vectorstore.similarity_search(query)

    print(f"\nSimilarity search for query: '{query}'")
    print("Found relevant documents:")
    for i, doc in enumerate(docs):
        print(f"\nDocument {i+1} (Page Content):\n{doc.page_content[:500]}...")

else:
    print("Document chunks or embeddings are not available. Please ensure previous steps were successful.")

Chroma vector store created successfully.

Similarity search for query: 'What are the licensing terms for Activepieces?'
Found relevant documents:

Document 1 (Page Content):
Activepieces License Compliance
Documentation
Reference: Oﬃcial License Documentation
Repository: activepieces/activepieces
This document maps the Activepieces dual-licensing model to this codebase and provides guidance
for compliance when using, modifying, or distributing the software.
1. License Summary
Component License Usage Terms
Core / Community
Edition MIT (Expat) Free to use, modify, and distribute.
Include license notice when distributing.
Enterprise / CloudCommercial
(packages/ee/LICEN...

Document 2 (Page Content):
License: MIT Expat (root  LICENSE )
Scope: All code outside  packages/ee/  and  packages/server/api/src/app/ee/ 
Rights: Use, modify, merge, publish, distribute, sublicense, sell
Obligation: Include copyright notice and license text when distributingValidation: MIT is widely used and commerci

In [30]:
query = "What is Asterisk used for?"

docs = vectorstore.similarity_search(query, k=3)

In [31]:
context = "\n\n".join([doc.page_content for doc in docs])

print("---- CONTEXT ----")
print(context)

---- CONTEXT ----
1.7 Am I Violating the EE License?
Use this decision ﬂow to determine if your use violates the EE license.EE License Violation Checklist
Scenario Violating
EE? Explanation
Using  AP_EDITION=ce 
(default) in production No CE mode does not enable EE features; EE code is
present but not used for commercial functionality
Using  AP_EDITION=ee  or
 cloud  in production
without a license
Yes EE license explicitly requires a valid subscription for
production use
Modifying EE code for
development or testing
only
No
EE license allows "copy and modify for
development and testing purposes without requiring
a subscription"
Modifying EE code and
using in production without
license
Yes Production use of EE (including modiﬁed EE)
requires a valid license
Distributing your fork that
includes unmodiﬁed EE
code
Risky
EE license forbids "copy, merge, publish, distribute,
sublicense, and/or sell" without a license. Including
EE in a public fork may be considered distribution.

Telemetry →

In [32]:
prompt = f"""
You are an AI assistant. Answer the question based only on the context below.

Context:
{context}

Question:
{query}

Answer:
"""

In [33]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",   # ✅ FIXED
    temperature=0.3
)

response = llm.invoke(prompt)

print(response.content)

The provided context does not mention what Asterisk is used for.


In [34]:
prompt = f"""
You are a helpful assistant.

Use ONLY the context below to answer the question.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question:
{query}

Answer in a clear and concise way:
"""

In [35]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

response = llm.invoke(prompt)

print(response.content)

I don't know.


In [36]:
query = "What is Asterisk used for?"

# Retrieve
docs = vectorstore.similarity_search(query, k=3)

# Build context
context = "\n\n".join([doc.page_content for doc in docs])

# Prompt
prompt = f"""
You are a helpful assistant.

Use ONLY the context below to answer the question.

Context:
{context}

Question:
{query}

Answer:
"""

# LLM call
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

response = llm.invoke(prompt)

print(response.content)

I apologize, but the provided context does not contain any information about what "Asterisk" is used for.
